# MyoVoice — Hardware Pipeline (ESP32 sEMG → 11-Word Classifier)

**Best Result: 88.87% accuracy on 11-word silent speech classification**

## Methodology
| Step | Details |
|---|---|
| **Hardware** | ESP32 microcontroller, 2-channel surface EMG sensors |
| **Signals** | `wire1_pin34` (primary), `wire2_pin32` (secondary) |
| **Preprocessing** | Mean centering, max-value normalization, silence gating |
| **Windowing** | Sliding window (25 samples, step 5) with activity threshold |
| **Features** | TD6: RMS, Waveform Length, Slope Sign Changes, Max, Min, Std |
| **Dual-channel** | Per-channel features concatenated → 12-dim feature vector |
| **Model** | Gradient Boosting Classifier (n=300, lr=0.1, depth=5) |
| **Vocabulary** | 11 words: START, STOP, TROUBLE, HELP, FALL, UP, ON, NO, DOWN, OFF, YES |
| **Dataset** | 18,648 windowed samples across 11 classes |
| **Accuracy** | **88.87%** on 20% held-out test set |

## Data
CSV files recorded from ESP32 hardware, one file per word.  
Expected columns: `timestamp_ms`, `wire1_pin34`, `wire2_pin32`  
Place files under `data/hardware/<WORD>.csv`

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from collections import Counter

print('Libraries loaded ✅')

In [ ]:
# ============================================================
# 2. CONFIGURATION
# Update DATA_DIR to point to your CSV folder.
# On Kaggle: '/kaggle/input/your-dataset/'
# Locally:   'data/hardware/'
# ============================================================
DATA_DIR = 'data/hardware/'   # <-- update this

# 11-word vocabulary and their CSV filenames
WORD_FILES = {
    'START':   os.path.join(DATA_DIR, 'START.csv'),
    'STOP':    os.path.join(DATA_DIR, 'STOP.csv'),
    'TROUBLE': os.path.join(DATA_DIR, 'TROUBLE.csv'),
    'HELP':    os.path.join(DATA_DIR, 'HELP.csv'),
    'FALL':    os.path.join(DATA_DIR, 'FALL.csv'),
    'UP':      os.path.join(DATA_DIR, 'UP.csv'),
    'ON':      os.path.join(DATA_DIR, 'ON.csv'),
    'NO':      os.path.join(DATA_DIR, 'NO.csv'),
    'DOWN':    os.path.join(DATA_DIR, 'DOWN.csv'),
    'OFF':     os.path.join(DATA_DIR, 'OFF.csv'),
    'YES':     os.path.join(DATA_DIR, 'YES.csv'),
}

# Sliding window parameters
WINDOW = 25   # samples per frame
STEP   = 5    # step between frames
ACTIVITY_THRESHOLD = 1.0  # silence gate (sum of |signal|)

print(f'Config: WINDOW={WINDOW}, STEP={STEP}, words={list(WORD_FILES.keys())}')

In [ ]:
# ============================================================
# 3. FEATURE EXTRACTION
# TD6 time-domain features per window:
#   RMS, Waveform Length, Slope Sign Changes, Max, Min, StdDev
# ============================================================
def extract_td6(signal: np.ndarray) -> list:
    """Extract 6 time-domain features from a 1D signal window."""
    rms = np.sqrt(np.mean(signal ** 2))          # Root Mean Square (power)
    wl  = np.sum(np.abs(np.diff(signal)))         # Waveform Length (complexity)
    diff = np.diff(signal)
    ssc = np.sum(np.diff(np.signbit(diff).astype(int)))  # Slope Sign Changes
    mx  = np.max(signal)                          # Max amplitude
    mn  = np.min(signal)                          # Min amplitude
    std = np.std(signal)                          # Standard deviation
    return [rms, wl, ssc, mx, mn, std]


def load_and_featurize(filepath: str, word: str) -> list:
    """
    Load a CSV file from the ESP32 hardware, extract dual-channel TD6 features.
    Returns list of 12-dim feature vectors.
    """
    if not os.path.exists(filepath):
        print(f'   ⚠️  File not found: {filepath}')
        return []

    try:
        df = pd.read_csv(filepath)

        # --- Smart column detection ---
        col1, col2 = None, None
        for c in df.columns:
            if 'wire1' in c.lower(): col1 = c
            if 'wire2' in c.lower(): col2 = c
        if col1 is None: col1 = df.columns[1]   # fallback: 2nd column
        if col2 is None: col2 = col1             # fallback: duplicate ch1

        # --- Load & normalize channel 1 ---
        raw1 = pd.to_numeric(df[col1], errors='coerce').fillna(0).values.astype(float)
        raw1 -= np.mean(raw1)
        if np.max(np.abs(raw1)) > 0: raw1 /= np.max(np.abs(raw1))

        # --- Load & normalize channel 2 (or duplicate if empty) ---
        raw2 = pd.to_numeric(df[col2], errors='coerce').values
        if np.isnan(raw2).all():
            raw2 = raw1.copy()
        else:
            raw2 = np.nan_to_num(raw2).astype(float)
            raw2 -= np.mean(raw2)
            if np.max(np.abs(raw2)) > 0: raw2 /= np.max(np.abs(raw2))

        # --- Sliding window feature extraction ---
        samples = []
        limit = min(len(raw1), len(raw2))
        for i in range(0, limit - WINDOW, STEP):
            c1 = raw1[i:i + WINDOW]
            c2 = raw2[i:i + WINDOW]
            if np.sum(np.abs(c1)) > ACTIVITY_THRESHOLD:  # silence gate
                samples.append(extract_td6(c1) + extract_td6(c2))  # 12-dim

        print(f'   ✅ {word:<8} | {len(samples):<5} samples | cols: {col1}, {col2}')
        return samples

    except Exception as e:
        print(f'   ❌ Error processing {word}: {e}')
        return []

print('Feature extractor defined ✅')

In [ ]:
# ============================================================
# 4. BUILD DATASET
# ============================================================
print('🏗️  Extracting features for all 11 words...\n')

X_all, y_all = [], []
for word, path in WORD_FILES.items():
    feats = load_and_featurize(path, word)
    X_all.extend(feats)
    y_all.extend([word] * len(feats))

X = np.array(X_all, dtype=np.float32)
y = np.array(y_all)

# Sanitize (replace any NaN/Inf)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print(f'\n📊 Dataset shape: X={X.shape}, classes={len(np.unique(y))}')
print(f'   Samples per class: {dict(Counter(y))}')

In [ ]:
# ============================================================
# 5. TRAIN / TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# ============================================================
# 6. TRAIN GRADIENT BOOSTING CLASSIFIER
# Best hyperparameters found after tuning:
#   n_estimators=300, learning_rate=0.1, max_depth=5
# ============================================================
print('🔧 Training Gradient Boosting Classifier...')

model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    verbose=1
)
model.fit(X_train, y_train)

print('\n✅ Training complete!')

In [ ]:
# ============================================================
# 7. EVALUATION
# ============================================================
preds = model.predict(X_test)
acc   = accuracy_score(y_test, preds)

print(f'\n🏆 FINAL TEST ACCURACY: {acc * 100:.2f}%')
print('\n📋 Classification Report:')
print(classification_report(y_test, preds))

In [ ]:
# ============================================================
# 8. CONFUSION MATRIX
# ============================================================
labels = sorted(np.unique(y))
cm = confusion_matrix(y_test, preds, labels=labels)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'MyoVoice — 11-Word Confusion Matrix (Acc: {acc * 100:.1f}%)')
plt.tight_layout()
plt.savefig('confusion_matrix_hardware.png', dpi=150)
plt.show()
print('Saved: confusion_matrix_hardware.png')

In [ ]:
# ============================================================
# 9. REAL-TIME INFERENCE (Sliding Window Voting)
# Use this block to run predictions on new recordings.
# ============================================================
def predict_word(csv_path: str, model, threshold: float = ACTIVITY_THRESHOLD) -> str:
    """
    Predict the spoken word from a new ESP32 recording.
    Uses majority voting across all active windows.
    Returns the predicted word and confidence %.
    """
    feats = load_and_featurize(csv_path, 'INPUT')
    if not feats:
        return 'No speech detected', 0.0

    X_input = np.nan_to_num(np.array(feats), nan=0.0)
    votes   = model.predict(X_input)
    counts  = Counter(votes)
    winner  = counts.most_common(1)[0][0]
    conf    = counts[winner] / len(votes) * 100

    return winner, conf


# --- Example usage ---
# pred, conf = predict_word('path/to/new_recording.csv', model)
# print(f'Predicted: {pred} ({conf:.1f}% confidence)')
print('Inference function defined ✅')

In [ ]:
# ============================================================
# 10. SAVE MODEL
# ============================================================
joblib.dump(model, 'myovoice_hardware_model.pkl')
print('💾 Model saved: myovoice_hardware_model.pkl')
print(f'\n✅ Final Summary')
print(f'   Model   : Gradient Boosting Classifier')
print(f'   Features: TD6 (RMS, WL, SSC, Max, Min, Std) × 2 channels = 12 dims')
print(f'   Vocab   : 11 words')
print(f'   Accuracy: {acc * 100:.2f}%')